# Delayed-entry survival model training

This notebook fits the four required
delayed-entry Cox candidates in every prespecified development scope, and records the forest and
boosting technical-admission decisions. It generates no assessment prediction, discrimination,
calibration, prediction-error, clinical-utility, explanation, or sensitivity result.


In [7]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from crc_folfox.model_training import run_model_training_workflow
from crc_folfox.model_training.admission import decide_boosting_admission, decide_forest_admission

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ANALYSIS_DIRECTORY = PROJECT_ROOT / "analysis" / "out" / "current"

forest_admission = decide_forest_admission()
boosting_admission = decide_boosting_admission()

result = run_model_training_workflow(
    ANALYSIS_DIRECTORY,
    conditional_model_admissions=(forest_admission, boosting_admission),
)

In [2]:
import json

assert result.artifacts is not None
manifest = json.loads(result.artifacts.training_manifest.read_text(encoding="utf-8"))
display(
    {"workflow": "delayed-entry model training", **{
        key: manifest[key]
        for key in ("input_hashes", "software_versions", "required_fit_count")
    }}
)

{'workflow': 'delayed-entry model training',
 'input_hashes': {'analysis_dataset.csv': '6b9e28099f547150324b33be900d228d22b411b7dbe0d68e1e08551aaab6740e',
  'data_dictionary.json': '74afd2302e93585bda59f4ecb01b536da05c0a44da14f07f66e8b660eeb27496',
  'feature_manifest.json': 'cc891bcdca2b06179dd33e4bc55160e0e048557059e57b0b02789c7b18dc0f73',
  'run_manifest.json': 'd11982f0e9cd7f740cae7432744fa23aba336d944f7236a2f10429e775b84bfd'},
 'software_versions': {'joblib': '1.5.3', 'lifelines': '0.30.3'},
 'required_fit_count': 424}

In [3]:
def _succeeded(status: pd.Series) -> int:
    return int((status == "fit_succeeded").sum())


inventory = (
    result.fit_summary.groupby(["endpoint", "design", "model_name"], sort=True)
    .agg(scopes=("model_id", "count"), succeeded=("fit_status", _succeeded))
    .reset_index()
)
display(inventory)

,endpoint,design,model_name,scopes,succeeded
0,OS,iecv_outer,core_cox,3,0
1,OS,iecv_outer,elastic_net_cox,3,1
2,OS,iecv_outer,minimal_cox,3,3
3,OS,iecv_outer,spline_cox,3,3
4,OS,pooled,core_cox,50,1
5,OS,pooled,elastic_net_cox,50,43
6,OS,pooled,minimal_cox,50,50
7,OS,pooled,spline_cox,50,50
8,TTNT-D,iecv_outer,core_cox,3,1
9,TTNT-D,iecv_outer,elastic_net_cox,3,3


## Required Cox candidates and conditional admission

`minimal_cox`  and `core_cox`  are unpenalized.
`elastic_net_cox` tunes the locked 12-point `penalizer` x `l1_ratio` grid on the core features.
`spline_cox` replaces the standardized linear `age_at_folfox` column with a fixed-ridge, four-knot
cubic spline basis on the core features. All four are required and use `lifelines.CoxPHFitter` with
explicit `duration_col="exit"`, `event_col="event"`, `entry_col="entry"`, Efron ties, and a Breslow
baseline. Every candidate passed its hand-calculated risk-set, Efron-likelihood, absolute-risk, and
entry-perturbation compatibility fixtures before any restricted-data fit.

The delayed-entry relative-risk forest (`ltrc_rrf`) and delayed-entry survival booster
(`ltrc_boosting`) join the candidate set only if they pass a nonblocking pre-result admission gate,
run once in isolated clean runtimes on synthetic data. Their frozen status for this run:

In [4]:
admissions = manifest["conditional_model_admissions"]
display(pd.DataFrame(admissions).T[["candidate", "status", "reason"]])

,candidate,status,reason
ltrc_boosting,ltrc_boosting,technically_ineligible,no_passing_pre_result_admission_gate: no dated...
ltrc_rrf,ltrc_rrf,technically_ineligible,preflight_failed: Required R forest compatibil...


## Common tuning objective and selected-parameter frequencies

Every tunable candidate minimizes the arithmetic mean of the delayed-entry IPCW Brier score at 365
and 730 days across the required inner-validation folds, with the censoring distribution fitted on
inner-training rows only. `elastic_net_cox` attempts all 12 locked `(penalizer, l1_ratio)`
combinations in every required inner fold; exact ties (within `1e-12`) favor the larger `penalizer`,
then the larger `l1_ratio`. This objective passed its own independent validation gate -- hand
fixtures, an independently coded reference, staggered-entry/censoring simulation, limiting cases,
and an entry-only perturbation sentinel -- before any restricted-data tuner ran.

In [6]:
selected = result.tuning_summary.query("selection_status == 'selected'")
frequency = (
    selected.groupby(["selected_penalizer", "selected_l1_ratio"])
    .size()
    .rename("scopes")
    .reset_index()
)
display(frequency.sort_values("scopes", ascending=False))

,selected_penalizer,selected_l1_ratio,scopes
5,0.100,0.0,42
6,0.100,0.5,40
0,0.001,0.0,3
4,0.010,1.0,3
7,1.000,0.0,3
2,0.010,0.0,2
3,0.010,0.5,2
1,0.001,1.0,1


## Spline configuration

`spline_cox`'s nonlinear term uses scikit-learn `SplineTransformer(degree=3, n_knots=4,
knots="quantile", include_bias=False, extrapolation="linear", order="C")`, fitted only on the
current narrowest training partition; the transformer replaces (never supplements) the standardized
linear `age_at_folfox` column, using a fixed ridge Cox penalty (`penalizer=0.01`, `l1_ratio=0.0`)
that is never tuned. No other column is splined.

The R forest runtime (`survival::Surv(entry, exit, event)`, `LTRCforests::ltrcrrf`) is pinned with
`renv` under `r/` and was exercised in repeated clean processes for the admission decision above; its
`technically_ineligible` status means no R prediction state was fitted for the required inventory.

## Final-fit completion, convergence, and warnings

A Cox fit succeeds only when the optimizer reports convergence and the complete fitted state is
finite. Warnings are captured per fit and classified into six declared aggregate classes; a warning
never changes a penalty, feature set, or model family.

In [6]:
completion = (
    result.fit_summary.groupby(["endpoint", "model_name", "fit_status", "convergence_status"])
    .size()
    .rename("count")
    .reset_index()
)
display(completion)

,endpoint,model_name,fit_status,convergence_status,count
0,OS,core_cox,fit_failed,not_converged,52
1,OS,core_cox,fit_succeeded,converged,1
2,OS,elastic_net_cox,fit_failed,not_converged,9
3,OS,elastic_net_cox,fit_succeeded,converged,43
4,OS,elastic_net_cox,fit_succeeded,converged_with_warning,1
5,OS,minimal_cox,fit_succeeded,converged,53
6,OS,spline_cox,fit_succeeded,converged,53
7,TTNT-D,core_cox,fit_failed,not_converged,51
8,TTNT-D,core_cox,fit_succeeded,converged,2
9,TTNT-D,elastic_net_cox,fit_failed,not_converged,1


## Canonical artifacts and privacy scan

The workflow atomically publishes `training_report.md`, `training_manifest.json`, `fit_summary.csv`,
`tuning_summary.csv`, and `model_registry.json` under `analysis/out/current/model_training/`, plus
one registered `.joblib` bundle per successful fit under its `models/` subdirectory. Every published
aggregate artifact and this notebook's own output contain no patient key, source identifier, exact
patient time, coefficient, baseline-hazard row, transformed row, prediction, or inner validation
score -- only the counts and identifiers displayed above.

In [7]:
display(
    pd.Series(
        {
            "training_report": result.artifacts.training_report.name,
            "training_manifest": result.artifacts.training_manifest.name,
            "fit_summary": result.artifacts.fit_summary.name,
            "tuning_summary": result.artifacts.tuning_summary.name,
            "model_registry": result.artifacts.model_registry.name,
            "directory": str(result.artifacts.directory),
        }
    )
)

training_report                                     training_report.md
training_manifest                               training_manifest.json
fit_summary                                            fit_summary.csv
tuning_summary                                      tuning_summary.csv
model_registry                                     model_registry.json
directory            /Users/camilasarmiento/Documents/Data Freelanc...
dtype: object